In [8]:
import pandas as pd
import pyarrow as pa

print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

arquivo = "05_PENSE2024_SEM_NAN_SEM_DUPLICADOS.parquet"
df = pd.read_parquet(arquivo)

print("Dataset dimensions:", df.shape)
print(df.dtypes)

Pandas: 3.0.6
PyArrow: 26.0.0
Dataset dimensions: (148725, 41)
REGIAO - código de regiões                                                                                                                                                                                                                                 string
UF - código UF                                                                                                                                                                                                                                             string
CAPITAL - indica se é município da capital                                                                                                                                                                                                                 string
B01001A - Qual é o seu sexo?                                                                                                                                       

In [9]:
import pandas as pd
import re

# ============================================================
# 1. ARQUIVOS
# ============================================================

arquivo_parquet = "05_PENSE2024_SEM_NAN_SEM_DUPLICADOS.parquet"
arquivo_excel = "01_Colunas_recomendadas_pense_2024_final.xlsx"
arquivo_saida = "06_PENSE2024_NUMERACAO_NOVA.parquet"


# ============================================================
# 2. LER OS ARQUIVOS
# ============================================================

df = pd.read_parquet(arquivo_parquet)

regras = pd.read_excel(
    arquivo_excel,
    header=1
)


# ============================================================
# 3. NOMES DAS COLUNAS DO EXCEL
# ============================================================

coluna_codigo = "Código da coluna"
coluna_regra = "Nova numeracao"

if coluna_codigo not in regras.columns:
    raise ValueError(
        f"Coluna '{coluna_codigo}' não encontrada no Excel."
    )

if coluna_regra not in regras.columns:
    raise ValueError(
        f"Coluna '{coluna_regra}' não encontrada no Excel."
    )


# ============================================================
# 4. FUNÇÃO PARA NORMALIZAR VALORES
# ============================================================
# O Parquet pode trazer:
#
#     1
#     1.0
#     "1"
#     "1.0"
#     " 1 "
#
# Todos passam a ser tratados como "1".
#
# Isso é importante porque as regras do Excel normalmente
# aparecem como 1, 2, -9, etc., enquanto o Parquet pode
# armazenar os mesmos códigos como 1.0, 2.0, -9.0.

def normalizar_valor(valor):

    if pd.isna(valor):
        return None

    texto = str(valor).strip()

    if texto == "":
        return None

    try:

        numero = float(texto)

        if numero.is_integer():
            return str(int(numero))

    except (ValueError, TypeError):
        pass

    return texto


# ============================================================
# 5. FUNÇÃO PARA CRIAR O MAPA DE CONVERSÃO
# ============================================================

def criar_mapa(regra):

    mapa = {}

    if pd.isna(regra):
        return mapa

    regra = str(regra).strip()

    # Remove parênteses que eventualmente apareçam no final
    regra = regra.rstrip(")")

    # Exemplos:
    #
    # 1=3,2=1,-9/-2/-1=0
    # 1/2=1,3=2,4/5=3,-9/-2/-1=0

    partes = regra.split(",")

    for parte in partes:

        parte = parte.strip()

        if not parte or "=" not in parte:
            continue

        origem, destino = parte.split("=", 1)

        origem = origem.strip()
        destino = destino.strip()

        valores_origem = origem.split("/")

        for valor in valores_origem:

            valor = valor.strip()

            if valor:

                mapa[normalizar_valor(valor)] = destino

    return mapa


# ============================================================
# 6. FUNÇÃO PARA ENCONTRAR A COLUNA NO PARQUET
# ============================================================

def encontrar_coluna(codigo, colunas_parquet):

    codigo = str(codigo).strip()

    # Primeiro: correspondência exata
    if codigo in colunas_parquet:
        return codigo

    # Segundo: código + " - "
    prefixo = codigo + " - "

    candidatas = [
        coluna
        for coluna in colunas_parquet
        if str(coluna).startswith(prefixo)
    ]

    if len(candidatas) == 1:
        return candidatas[0]

    if len(candidatas) > 1:

        print(
            f"ATENÇÃO: mais de uma coluna encontrada "
            f"para o código '{codigo}':"
        )

        for coluna in candidatas:
            print(f"    {coluna}")

        return None

    return None


# ============================================================
# 7. DIAGNÓSTICO DOS TIPOS DO PARQUET
# ============================================================

print("=" * 80)
print("TIPOS DAS COLUNAS DO PARQUET")
print("=" * 80)

print(df.dtypes.to_string())

print()
print("=" * 80)
print("EXEMPLO DOS VALORES REAIS")
print("=" * 80)

for coluna in df.columns[:10]:

    print()
    print(f"COLUNA: {coluna}")
    print(f"TIPO:   {df[coluna].dtype}")
    print(
        "VALORES:",
        df[coluna].drop_duplicates().head(10).tolist()
    )


# ============================================================
# 8. PROCESSAR TODAS AS REGRAS
# ============================================================

relatorio = []
diagnostico = []

colunas_processadas = 0
total_alteracoes = 0
total_correspondencias = 0


for _, linha in regras.iterrows():

    codigo = linha[coluna_codigo]

    if pd.isna(codigo):
        continue

    codigo = str(codigo).strip()

    regra = linha[coluna_regra]

    if pd.isna(regra):
        continue

    regra = str(regra).strip()

    if regra == "":
        continue


    # --------------------------------------------------------
    # Encontrar a coluna real
    # --------------------------------------------------------

    coluna_real = encontrar_coluna(
        codigo,
        df.columns
    )

    if coluna_real is None:

        print(
            f"AVISO: código '{codigo}' "
            f"não encontrado no Parquet."
        )

        relatorio.append({
            "codigo_excel": codigo,
            "coluna_parquet": "",
            "regra": regra,
            "status": "NÃO ENCONTRADA",
            "correspondencias": 0,
            "alteracoes": 0
        })

        continue


    # --------------------------------------------------------
    # Criar mapa
    # --------------------------------------------------------

    mapa = criar_mapa(regra)

    if not mapa:

        relatorio.append({
            "codigo_excel": codigo,
            "coluna_parquet": coluna_real,
            "regra": regra,
            "status": "REGRA VAZIA/INVÁLIDA",
            "correspondencias": 0,
            "alteracoes": 0
        })

        continue


    # --------------------------------------------------------
    # Normalizar os valores da coluna SOMENTE para a busca
    # --------------------------------------------------------
    # Não usamos replace() diretamente porque:
    #
    #     1.0 != 1
    #
    # A série normalizada transforma ambos em "1".

    valores_antes = df[coluna_real].copy()

    valores_normalizados = (
        valores_antes
        .map(normalizar_valor)
    )


    # --------------------------------------------------------
    # Identificar quais valores realmente existem
    # e quais deles possuem regra
    # --------------------------------------------------------

    frequencias = (
        valores_normalizados
        .value_counts(dropna=False)
    )

    correspondencias = 0

    for valor, quantidade in frequencias.items():

        if valor in mapa:
            correspondencias += int(quantidade)

            diagnostico.append({
                "codigo_excel": codigo,
                "coluna_parquet": coluna_real,
                "valor_encontrado": valor,
                "quantidade": int(quantidade),
                "novo_valor": mapa[valor]
            })


    # --------------------------------------------------------
    # Aplicar a transformação
    # --------------------------------------------------------

    valores_novos = valores_normalizados.map(mapa)

    # Valores sem regra permanecem exatamente como estavam
    valores_novos = valores_novos.fillna(valores_antes)

    # Como as novas categorias são numéricas, tentamos manter
    # a coluna numérica quando isso for possível.
    convertido_numerico = pd.to_numeric(
        valores_novos,
        errors="coerce"
    )

    if convertido_numerico.notna().all():

        valores_novos = convertido_numerico


    df[coluna_real] = valores_novos


    # --------------------------------------------------------
    # Contar alterações reais
    # --------------------------------------------------------

    alteracoes = (
        valores_antes.astype("string")
        != df[coluna_real].astype("string")
    ).sum()

    alteracoes = int(alteracoes)

    total_correspondencias += correspondencias
    total_alteracoes += alteracoes
    colunas_processadas += 1


    # --------------------------------------------------------
    # Relatório
    # --------------------------------------------------------

    status = "ALTERADO"

    if correspondencias == 0:
        status = "NENHUMA CORRESPONDÊNCIA"

    relatorio.append({
        "codigo_excel": codigo,
        "coluna_parquet": coluna_real,
        "regra": regra,
        "status": status,
        "correspondencias": correspondencias,
        "alteracoes": alteracoes
    })


    # --------------------------------------------------------
    # Mostrar progresso
    # --------------------------------------------------------

    print(
        f"{status} | {codigo} | "
        f"correspondências: {correspondencias:,} | "
        f"alterações: {alteracoes:,}"
    )


# ============================================================
# 9. SALVAR O NOVO PARQUET
# ============================================================

df.to_parquet(
    arquivo_saida,
    engine="pyarrow",
    compression="snappy",
    index=False
)


# ============================================================
# 10. RELATÓRIOS
# ============================================================

relatorio_df = pd.DataFrame(relatorio)
diagnostico_df = pd.DataFrame(diagnostico)


# Relatório geral
relatorio_df.to_excel(
    "05_RELATORIO_RECODIFICACAO.xlsx",
    index=False
)

# Diagnóstico detalhado:
# mostra exatamente quais valores foram encontrados
# e para qual valor cada um foi recodificado.
diagnostico_df.to_excel(
    "05_DIAGNOSTICO_RECODIFICACAO.xlsx",
    index=False
)


# ============================================================
# 11. RESULTADO FINAL
# ============================================================

print()
print("=" * 80)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 80)

print(
    f"Colunas processadas:          {colunas_processadas}"
)

print(
    f"Total de correspondências:    {total_correspondencias:,}"
)

print(
    f"Total de valores alterados:   {total_alteracoes:,}"
)

print(
    f"Arquivo criado:               {arquivo_saida}"
)

print(
    "Relatório geral:              "
    "05_RELATORIO_RECODIFICACAO.xlsx"
)

print(
    "Diagnóstico detalhado:        "
    "05_DIAGNOSTICO_RECODIFICACAO.xlsx"
)

print()
print("=" * 80)
print("RELATÓRIO FINAL")
print("=" * 80)

display(relatorio_df)

print()
print("=" * 80)
print("DIAGNÓSTICO DOS VALORES ENCONTRADOS")
print("=" * 80)

display(diagnostico_df)


TIPOS DAS COLUNAS DO PARQUET
REGIAO - código de regiões                                                                                                                                                                                                                                 string
UF - código UF                                                                                                                                                                                                                                             string
CAPITAL - indica se é município da capital                                                                                                                                                                                                                 string
B01001A - Qual é o seu sexo?                                                                                                                                                                         

,codigo_excel,coluna_parquet,regra,status,correspondencias,alteracoes
0,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...","1=3,2=1,-9/-2/-1=0",ALTERADO,148725,148725
1,E01P117,"E01P117 - NOS ÚLTIMOS 12 MESES, a escola conto...","1=1,2=2,3=3,4=4,-9/-2/-1=0",ALTERADO,148725,188
2,E01P11801,E01P11801 - Quais tipos de suporte a escola of...,"1=3,0=1,-9/-2/-1=0)",ALTERADO,148725,148725
3,E01P11802,E01P11802 - Quais tipos de suporte a escola of...,"1=3,0=1,-9/-2/-1=0)",ALTERADO,148725,148725
4,E01P8505,"E01P8505 - NOS ÚLTIMOS 12 MESES, a escola dese...","1=3,0=1,-9/-2/-1=0)",ALTERADO,148725,148725
5,E01P8510,"E01P8510 - NOS ÚLTIMOS 12 MESES, a escola dese...","1=3,0=1,-9/-2/-1=0)",ALTERADO,148725,148725
6,B07006,"B07006 - NOS ÚLTIMOS 30 DIAS, com que frequênc...","1/2=1,3=2,4/5=3,-9/-2/-1=0",ALTERADO,148725,138918
7,B07007B,"B07007B - NOS ÚLTIMOS 30 DIAS, quantas vezes a...","1=1,2/3/4=3,-9/-2/-1=0",ALTERADO,148725,48650
8,B07009,"B07009 - NOS ÚLTIMOS 30 DIAS, você esculachou,...","1=3,2=1,-9/-2/-2=0",ALTERADO,148725,148725
9,B07011A,"B07011A - NOS ÚLTIMOS 30 DIAS, quantas vezes a...","1=1,2/3/4=3,-9/-2/-1=0",ALTERADO,148725,32289



DIAGNÓSTICO DOS VALORES ENCONTRADOS


,codigo_excel,coluna_parquet,valor_encontrado,quantidade,novo_valor
0,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...",2,99276,1
1,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...",1,48563,3
2,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...",-9,527,0
3,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...",-2,359,0
4,E01P117,"E01P117 - NOS ÚLTIMOS 12 MESES, a escola conto...",1,76045,1
...,...,...,...,...,...
218,B05002A,B05002A - Alguma vez na vida você tomou um cop...,1,65920,3
219,B05002A,B05002A - Alguma vez na vida você tomou um cop...,2,58653,1
220,B05002A,B05002A - Alguma vez na vida você tomou um cop...,-1,23831,0
221,B05002A,B05002A - Alguma vez na vida você tomou um cop...,-9,165,0


In [10]:
import pandas as pd
from IPython.display import HTML, display


# ============================================================
# 1. ARQUIVO COM AS REGRAS DE RECODIFICAÇÃO
# ============================================================

arquivo_excel = "01_Colunas_recomendadas_pense_2024_final.xlsx"


# ============================================================
# 2. LER O ARQUIVO EXCEL
# ============================================================

regras = pd.read_excel(
    arquivo_excel,
    header=1
)


# ============================================================
# 3. NOMES DAS COLUNAS DO EXCEL
# ============================================================

coluna_codigo = "Código da coluna"
coluna_codificacao = "Nova codificacao"


if coluna_codigo not in regras.columns:
    raise ValueError(
        f"Coluna '{coluna_codigo}' não encontrada no Excel."
    )


if coluna_codificacao not in regras.columns:
    raise ValueError(
        f"Coluna '{coluna_codificacao}' não encontrada no Excel."
    )


# ============================================================
# 4. FUNÇÃO PARA NORMALIZAR OS CÓDIGOS
# ============================================================
# Isso permite comparar corretamente:
#
# 1
# 1.0
# "1"
# "1.0"
# " 1 "
#
# Todos passam a ser tratados como "1".

def normalizar_codigo(valor):

    if pd.isna(valor):
        return None

    texto = str(valor).strip()

    if texto == "":
        return None

    try:

        numero = float(texto)

        if numero.is_integer():
            return str(int(numero))

    except (ValueError, TypeError):

        pass

    return texto


# ============================================================
# 5. FUNÇÃO PARA CRIAR O MAPA DA "NOVA CODIFICACAO"
# ============================================================
#
# Exemplo:
#
# 3=Sim
# 1=Nao
# 0=Sem resposta
#
# vira:
#
# {
#     "3": "Sim",
#     "1": "Nao",
#     "0": "Sem resposta"
# }

def criar_mapa_codificacao(codificacao):

    mapa = {}

    if pd.isna(codificacao):
        return mapa

    texto = str(codificacao).strip()

    if texto == "":
        return mapa


    # A codificação pode estar separada por vírgulas
    # ou por quebras de linha.

    partes = re.split(
        r",|\n",
        texto
    )


    for parte in partes:

        parte = parte.strip()

        if not parte:
            continue


        # Procura o primeiro "="

        if "=" not in parte:
            continue


        codigo, legenda = parte.split(
            "=",
            1
        )


        codigo = normalizar_codigo(codigo)

        legenda = legenda.strip()


        if codigo is not None and legenda:

            mapa[codigo] = legenda


    return mapa


# ============================================================
# 6. CRIAR O MAPA:
#
# CÓDIGO DA COLUNA
#        ↓
# NOVA CODIFICACAO
# ============================================================

mapas_codificacao = {}


for _, linha in regras.iterrows():

    codigo = linha[coluna_codigo]

    if pd.isna(codigo):
        continue

    codigo = str(codigo).strip()


    codificacao = linha[coluna_codificacao]

    if pd.isna(codificacao):
        continue


    mapa = criar_mapa_codificacao(
        codificacao
    )


    if mapa:

        mapas_codificacao[codigo] = mapa


# ============================================================
# 7. FUNÇÃO PARA ENCONTRAR A COLUNA NO PARQUET
# ============================================================

def encontrar_coluna(codigo, colunas_parquet):

    codigo = str(codigo).strip()


    # Correspondência exata

    if codigo in colunas_parquet:

        return codigo


    # Procura pelo código no início do nome da coluna
    #
    # Exemplo:
    #
    # B12009
    #
    # B12009 - NOS ÚLTIMOS 12 MESES...

    prefixo = codigo + " - "


    candidatas = [
        coluna
        for coluna in colunas_parquet
        if str(coluna).startswith(prefixo)
    ]


    if len(candidatas) == 1:

        return candidatas[0]


    return None


# ============================================================
# 8. GERAR A TABELA RESUMO
# ============================================================

tabela_dados = []


for col in df.columns:

    col_limpa = str(col).strip()


    # ========================================================
    # IDENTIFICAR O CÓDIGO DA VARIÁVEL
    # ========================================================

    if " - " in col_limpa:

        codigo_variavel, descricao_variavel = (
            col_limpa.split(
                " - ",
                1
            )
        )

        codigo_variavel = codigo_variavel.strip()
        descricao_variavel = descricao_variavel.strip()

    else:

        codigo_variavel = col_limpa
        descricao_variavel = col_limpa


    # ========================================================
    # PROCURAR A CODIFICAÇÃO NO ARQUIVO 01
    # ========================================================

    mapa = mapas_codificacao.get(
        codigo_variavel,
        {}
    )


    # ========================================================
    # CONTAR AS RESPOSTAS
    # ========================================================

    counts = df[col].value_counts(
        dropna=False
    )


    total = len(df)


    respostas_formatadas = []


    # ========================================================
    # FORMATAR CADA RESPOSTA
    # ========================================================

    for valor, quantidade in counts.items():

        pct = (
            quantidade / total
        ) * 100


        # ----------------------------------------------------
        # VALORES NULOS / EM BRANCO
        # ----------------------------------------------------

        if pd.isna(valor):

            respostas_formatadas.append(
                f"[NaN] Valores Nulos / Em Branco (NaN): "
                f"{pct:.2f}%"
            )

            continue


        # ----------------------------------------------------
        # NORMALIZAR O VALOR
        # ----------------------------------------------------

        codigo_resposta = normalizar_codigo(
            valor
        )


        # ----------------------------------------------------
        # PROCURAR A LEGENDA NA "NOVA CODIFICACAO"
        # ----------------------------------------------------

        legenda = mapa.get(
            codigo_resposta
        )


        # ----------------------------------------------------
        # MONTAR O RESULTADO
        # ----------------------------------------------------

        if legenda is not None:

            respostas_formatadas.append(
                f"[{codigo_resposta}] "
                f"{pct:.2f}% - {legenda}"
            )

        else:

            # Se não houver legenda no arquivo 01,
            # mantém apenas o código e percentual.

            respostas_formatadas.append(
                f"[{codigo_resposta}] "
                f"{pct:.2f}%"
            )


    # ========================================================
    # ADICIONAR À TABELA FINAL
    # ========================================================

    tabela_dados.append({

        "Pergunta / Variável":
            col_limpa,

        "Distribuição das Respostas":
            "\n".join(
                respostas_formatadas
            )
    })


# ============================================================
# 9. CRIAR DATAFRAME DA TABELA
# ============================================================

df_tabela = pd.DataFrame(
    tabela_dados
)


# ============================================================
# 10. ESTILIZAR A TABELA
# ============================================================

styler = df_tabela.style.set_properties(
    **{
        "white-space": "pre-wrap",
        "text-align": "left"
    }
)


# ============================================================
# 11. EXIBIR
# ============================================================

print(
    "========== TABELA FINAL DOS RESULTADOS ==========\n"
)

display(
    HTML(
        styler.to_html()
    )
)

========== TABELA FINAL DOS RESULTADOS ==========



,Pergunta / Variável,Distribuição das Respostas
0,REGIAO - código de regiões,[2] 33.43% - nordeste [1] 24.48% - norte [3] 18.45% - sudeste [5] 12.30% - centro-oeste [4] 11.33% - sul
1,UF - código UF,[32] 4.95% - Espírito Santo [35] 4.75% - São Paulo [26] 4.54% - Pernambuco [29] 4.53% - Bahia [33] 4.50% - Rio de Janeiro [11] 4.31% - Rondônia [31] 4.26% - Minas Gerais [41] 4.16% - Paraná [17] 4.08% - Tocantins [42] 4.03% - Santa Catarina [50] 3.96% - Mato Grosso do Sul [21] 3.94% - Maranhão [28] 3.85% - Sergipe [27] 3.79% - Alagoas [13] 3.58% - Amazonas [15] 3.50% - Pará [25] 3.49% - Paraíba [22] 3.28% - Piauí [12] 3.26% - Acre [16] 3.18% - Amapá [43] 3.14% - Rio Grande do Sul [23] 3.03% - Ceará [24] 2.99% - Rio Grande do Norte [51] 2.95% - Mato Grosso [52] 2.90% - Goiás [14] 2.58% - Roraima [53] 2.50% - Distrito Federal
2,CAPITAL - indica se é município da capital,[1] 51.21% - Capital [0] 48.79%
3,B01001A - Qual é o seu sexo?,[2] 50.05% - Mulher [1] 49.95% - Homem
4,"IDADE_AGREG - IDADE DO ESTUDANTE, CATEGORIAS",[2] 51.62% - 13 a 15 anos [3] 27.75% - 16 e 17 anos [1] 16.03% - Menos de 13 anos [4] 4.52% - 18 anos ou mais [0] 0.08% - Sem resposta
5,B01002 - Qual é a sua cor ou raça?,[4] 43.22% - Parda [1] 40.53% - Branca [2] 10.67% - Preta [3] 2.75% - Amarela [5] 2.25% - Indígena [0] 0.58% - Sem reposta
6,B01006 - Você mora com sua mãe?,[3] 89.93% - Sim [1] 10.02% - Nao [0] 0.04% - Sem resposta
7,B01007 - Você mora com seu pai?,[3] 63.45% - Sim [1] 36.47% - Nao [0] 0.08% - Sem resposta
8,B01015B - Na sua casa tem computador ou notebook?,[3] 63.27% - Sim [1] 36.67% - Nao [0] 0.06% - Sem resposta
9,B01016 - Você tem acesso à internet em sua casa?,[3] 97.26% - Sim [1] 2.69% - Nao [0] 0.05% - Sem resposta
